# YOLO model local training

This notebook shows how to train a YOLO model locally. It also shows all the hyperparameters that can be tuned to control aspects related to the complexity and learning rate of the model, as well as stoping criteria.

In [1]:
from IPython import display
display.clear_output()

import cv2
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import roboflow

import ultralytics
from ultralytics import YOLO
ultralytics.checks()

Ultralytics 8.4.33 🚀 Python-3.12.8 torch-2.6.0 CPU (Apple M4 Pro)
Setup complete ✅ (14 CPUs, 24.0 GB RAM, 796.1/926.4 GB disk)


In [2]:
# Em apple sillicon, verificar se o MPS está disponível
import torch
print(torch.backends.mps.is_available())

True


## Add tensorboard support
In your terminal run tensorboard --logdir runs/train and then open http://localhost:6006 in your browser. You'll see:

- Scalars: loss curves (box, cls, dfl), mAP, precision, recall per epoch
- Images: sample validation predictions with bounding boxes
- Graphs: the model architecture (if enabled)

Also, in your console, check that tensorboard is set to true, by running `yolo settings`. If it's set to false, run `yolo settings tensorboard=True`

In [3]:
from torch.utils.tensorboard import SummaryWriter
from ultralytics.utils.callbacks.tensorboard import callbacks as tb_callbacks

# Launch TensorBoard (run this in your terminal, pointing to your project folder):
# tensorboard --logdir runs/detect/runs/train/foe-bot-exp1

# TensorBoard is enabled automatically by Ultralytics when tensorboard is installed.
# Just make sure it's installed:
# pip install tensorboard

## Download dataset

In [4]:
# Download the dataset from Roboflow, after labeling the images and creating the dataset.
# On Mac, permission issues may arise — fix them with:
# - sudo mkdir /Users/<your-username>/.config/roboflow   (create the config folder)
# - sudo chown -R <your-username>:staff ~/.config/roboflow  (grant your user ownership)
roboflow.login()

# Create a file containing your Roboflow API key, or replace the variable directly below
with open('api_key', "r") as file:
    api_key = file.read().strip()

rf = roboflow.Roboflow(api_key)

# Replace with your own workspace and project names
project = rf.workspace("dcarneiro").project("foe-bot")
# If the dataset version is > 1, replace with the corresponding version number
dataset = project.version(2).download("yolov8")
# WARN: After downloading, verify that the paths inside data.yaml are correct

You are already logged into Roboflow. To make a different login,run roboflow.login(force=True).
loading Roboflow workspace...
loading Roboflow project...


## Configuring the hyperparameters and training the model
Most impactful parameters to tune first: epochs, batch, imgsz, lr0, and freeze (if fine-tuning). The augmentation defaults are generally solid and rarely need changing unless your dataset is very small or domain-specific.

Apple Silicon note: device="mps" works well for YOLOv8 but amp=True can occasionally cause instability — if you see NaN losses, set amp=False.

Multi-GPU (Windows/Linux): Replace device='mps' with device=[0, 1] and consider increasing batch proportionally (e.g. 2 GPUs → double the batch size).

Early stopping tip: Setting patience=20–50 is a good practice to avoid wasting compute once the model has converged.

In [ ]:
# Load the model
# Full list of available pre-trained models: https://docs.ultralytics.com/models/yolov8/#performance-metrics
model = YOLO("yolov8m.pt")  # Load the pre-trained model weights

# ─────────────────────────────────────────────────────────────────────────────
# model.train() — Full hyperparameter reference
# ─────────────────────────────────────────────────────────────────────────────
results = model.train(

    # ── Dataset ──────────────────────────────────────────────────────────────
    data='FoE-bot-2/data.yaml',  # Path to the dataset config file (YAML).
                                  # Must contain train/val/test paths and class names.

    # ── Core training settings ────────────────────────────────────────────────
    epochs=100,         # Total number of training epochs.
                        # More epochs = more learning, but risk of overfitting.
                        # Typical range: 50–300.

    patience=50,        # Early stopping: halt training if no improvement is seen
                        # for this many epochs. Set to 0 to disable.

    batch=16,           # Number of images per training batch.
                        # Higher = faster but needs more VRAM. Use -1 for AutoBatch.

    imgsz=640,          # Input image size (pixels). Images are resized to this square.
                        # Common values: 416, 512, 640, 1280.

    # ── Hardware ──────────────────────────────────────────────────────────────
    device="mps",       # Device to train on.
                        # "cpu"       → CPU only (slow)
                        # 0           → first CUDA GPU
                        # [0, 1]      → multi-GPU (CUDA)
                        # "mps"       → Apple Silicon GPU

    workers=8,          # Number of DataLoader worker threads for loading images.
                        # Reduce if you see memory errors or CPU bottlenecks.

    # ── Checkpointing & output ────────────────────────────────────────────────
    project="runs/train",   # Root folder where training results are saved.
    name="foe-bot-exp1",    # Sub-folder name for this specific run.
                            # Results go to <project>/<name>/.

    exist_ok=False,     # If False, a new numbered folder is created when the name
                        # already exists. If True, the existing folder is overwritten.

    save=True,          # Save checkpoints (best.pt and last.pt) during training.
    save_period=-1,     # Save a checkpoint every N epochs. -1 = disabled.
                        # Example: save_period=10 saves every 10 epochs.

    # ── Transfer learning ─────────────────────────────────────────────────────
    pretrained=True,    # Start from pre-trained weights (recommended).
                        # Set to False to train from scratch.

    freeze=0,           # Freeze the first N layers of the backbone (do not update
                        # their weights). Useful for fine-tuning.
                        # Example: freeze=10 freezes the first 10 layers.

    # ── Optimiser ─────────────────────────────────────────────────────────────
    optimizer="auto",   # Optimiser algorithm. Options:
                        # "SGD", "Adam", "AdamW", "NAdam", "RAdam", "RMSProp", "auto"
                        # "auto" selects SGD or AdamW based on the model.

    lr0=0.01,           # Initial learning rate.
                        # Lower values (e.g. 0.001) are safer for fine-tuning.

    lrf=0.01,           # Final learning rate as a fraction of lr0.
                        # The scheduler decays lr0 → lr0 * lrf over training.

    momentum=0.937,     # SGD momentum / Adam beta1.
                        # Controls how much past gradients influence the update.

    weight_decay=0.0005, # L2 regularisation penalty — discourages large weights
                         # and helps prevent overfitting.

    warmup_epochs=3.0,  # Number of epochs for learning-rate warm-up at the start.
                        # LR gradually rises from 0 to lr0 during this phase.

    warmup_momentum=0.8, # Initial momentum during the warm-up phase.
    warmup_bias_lr=0.1,  # Initial learning rate for bias parameters during warm-up.

    # ── Loss weights ──────────────────────────────────────────────────────────
    box=7.5,            # Weight for the bounding-box regression loss.
                        # Increase to make the model focus more on box accuracy.

    cls=0.5,            # Weight for the classification loss.
    dfl=1.5,            # Weight for the Distribution Focal Loss (box refinement).

    # ── Augmentation ──────────────────────────────────────────────────────────
    hsv_h=0.015,        # Random hue shift (fraction of 360°). Adds colour variety.
    hsv_s=0.7,          # Random saturation shift. Range: 0.0–1.0.
    hsv_v=0.4,          # Random brightness (value) shift. Range: 0.0–1.0.

    degrees=0.0,        # Random rotation range in degrees (e.g. 10 → ±10°).
    translate=0.1,      # Random translation as a fraction of image size (e.g. 0.1 = 10%).
    scale=0.5,          # Random scale factor (e.g. 0.5 means zoom between 50%–150%).
    shear=0.0,          # Random shear angle in degrees.
    perspective=0.0,    # Random perspective distortion. Range: 0.0–0.001.
    flipud=0.0,         # Probability of vertical flip. 0.0 = disabled.
    fliplr=0.5,         # Probability of horizontal flip. 0.5 = 50% chance per image.

    mosaic=1.0,         # Probability of Mosaic augmentation (combines 4 images).
                        # Very effective for small object detection. 0.0 = disabled.

    mixup=0.0,          # Probability of MixUp augmentation (blends 2 images).
                        # Useful for improving generalisation.

    copy_paste=0.0,     # Probability of Copy-Paste augmentation (pastes objects
                        # from one image onto another). Good for instance segmentation.

    # ── Evaluation ────────────────────────────────────────────────────────────
    val=True,           # Run validation after each epoch to track mAP, loss, etc.

    plots=True,         # Generate and save training plots (loss curves, confusion
                        # matrix, PR curve) inside the results folder.

    # ── Reproducibility ───────────────────────────────────────────────────────
    seed=0,             # Random seed for reproducibility. Use the same seed to get
                        # identical results across runs.

    deterministic=True, # Force deterministic CUDA operations.
                        # Ensures reproducibility but may slow down training slightly.

    # ── Misc ──────────────────────────────────────────────────────────────────
    resume=False,       # Resume training from the last saved checkpoint (last.pt).
                        # Set to the checkpoint path string to resume a specific run.

    amp=False,           # Automatic Mixed Precision (FP16). Speeds up training and
                        # reduces VRAM usage on supported hardware.

    fraction=1.0,       # Fraction of the training dataset to use.
                        # Example: 0.1 uses only 10% of images (useful for quick tests).

    profile=False,      # Profile ONNX and TensorRT speeds during training.
                        # Useful for benchmarking deployment targets.

    verbose=True,       # Print detailed training logs to the console.
      3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
      4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
)

Ultralytics 8.4.33 🚀 Python-3.12.8 torch-2.6.0 MPS (Apple M4 Pro)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=FoE-bot-2/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=foe-bot-exp12, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=50, perspective=0.0, plots=True, pose

The input to trace is already a ScriptModule, tracing it is a no-op. Returning the object as is.


TensorBoard: model graph visualization added ✅
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /Users/davidecarneiro/Library/CloudStorage/Dropbox/Trabalho/Aulas/2025 - 2026/LEI - IA/TP AC/ia_25_26/runs/detect/runs/train/foe-bot-exp12
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      14.4G       1.86      4.523      1.326         17        640: 100% ━━━━━━━━━━━━ 2/2 23.5s/it 47.0s.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.1s/it 1.1s
                   all          5         30      0.371      0.111      0.028     0.0132

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      13.7G      2.104      3.851      1.298         26        640: 100% ━━━━━━━━━━━━ 2/2 1.9s/it 3.8s3.2s
WARNING ⚠️ NMS time limit 2.250s exceeded
                 Class     Images  Instances      

# Inference

Now we'll use the trained model to predict on images on a folder. This can eventually be changed to aquire images from a camera in real time, or even for video. You will need to develop an app to showcase your usecase, and it can be based on this code. 

In [10]:
import json
import os
from pathlib import Path

import cv2
from ultralytics import YOLO

# ─────────────────────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────────────────────
MODEL_PATH   = "runs/detect/runs/train/foe-bot-exp12/weights/best.pt"
INPUT_DIR    = Path("input")
OUTPUT_DIR   = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}

# ─────────────────────────────────────────────────────────────────────────────
# Load the trained model
# ─────────────────────────────────────────────────────────────────────────────
model = YOLO(MODEL_PATH)

# ─────────────────────────────────────────────────────────────────────────────
# Run inference on each image
# ─────────────────────────────────────────────────────────────────────────────
image_paths = [
    p for p in INPUT_DIR.iterdir()
    if p.suffix.lower() in SUPPORTED_EXTENSIONS
]

if not image_paths:
    print(f"No images found in '{INPUT_DIR}'. Supported formats: {SUPPORTED_EXTENSIONS}")

for image_path in image_paths:
    print(f"Processing: {image_path.name}")

    # ── Run prediction ────────────────────────────────────────────────────────
    results = model.predict(
        source=str(image_path),
        conf=0.25,          # Minimum confidence threshold to consider a detection.
                            # Detections below this value are discarded.
                            # Range: 0.0–1.0. Lower = more detections, more noise.

        iou=0.7,            # IoU threshold for Non-Maximum Suppression (NMS).
                            # Overlapping boxes above this threshold are merged.
                            # Lower = stricter deduplication.

        imgsz=640,          # Resize input to this size before inference.
                            # Should match the size used during training.

        max_det=300,        # Maximum number of detections per image.

        device="mps",       # Inference device. Use "cpu", 0, [0,1], or "mps".

        verbose=False,      # Suppress per-image console output.
    )

    result = results[0]  # One result per image

    # ── Build the detections JSON ─────────────────────────────────────────────
    detections = []
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()   # Bounding box corners (pixels)
        detections.append({
            "class_id":   int(box.cls),                    # Numeric class index
            "class_name": model.names[int(box.cls)],       # Human-readable class label
            "confidence": round(float(box.conf), 4),       # Detection confidence score
            "bbox": {
                "x1": round(x1, 2),
                "y1": round(y1, 2),
                "x2": round(x2, 2),
                "y2": round(y2, 2),
                "width":  round(x2 - x1, 2),
                "height": round(y2 - y1, 2),
            }
        })

    # ── Save JSON file ────────────────────────────────────────────────────────
    json_path = OUTPUT_DIR / f"{image_path.stem}.json"
    with open(json_path, "w") as f:
        json.dump(detections, f, indent=2)

    # ── Save annotated image ──────────────────────────────────────────────────
    # result.plot() returns a BGR numpy array with bounding boxes drawn on it
    annotated_image = result.plot(
        conf=True,      # Show confidence score on each label
        labels=True,    # Show class name on each box
        boxes=True,     # Draw bounding boxes
        line_width=2,   # Box border thickness in pixels
    )
    output_image_path = OUTPUT_DIR / image_path.name
    cv2.imwrite(str(output_image_path), annotated_image)

    print(f"  → {len(detections)} detection(s) | image saved to '{output_image_path}' | JSON saved to '{json_path}'")

print("\nDone.")

Processing: IMG_1.jpg
  → 4 detection(s) | image saved to 'output/IMG_1.jpg' | JSON saved to 'output/IMG_1.json'
Processing: IMG_2.jpg
  → 6 detection(s) | image saved to 'output/IMG_2.jpg' | JSON saved to 'output/IMG_2.json'
Processing: IMG_3.jpg
  → 22 detection(s) | image saved to 'output/IMG_3.jpg' | JSON saved to 'output/IMG_3.json'
Processing: IMG_4.jpg
  → 2 detection(s) | image saved to 'output/IMG_4.jpg' | JSON saved to 'output/IMG_4.json'
Processing: IMG_5.jpg
  → 3 detection(s) | image saved to 'output/IMG_5.jpg' | JSON saved to 'output/IMG_5.json'

Done.
